In [1]:
# ── BLOCK 1: Imports & Load IEEE-CIS Dataset ──
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import (f1_score, accuracy_score, 
                             roc_auc_score, classification_report)
from imblearn.over_sampling import SMOTE
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# Load IEEE-CIS transaction data
print("Loading IEEE-CIS dataset...")
train_transaction = pd.read_csv(
    r"E:\NCI\Sem3\Thesis\implementation\data\ieee-cis\train_transaction.csv")
train_identity = pd.read_csv(
    r"E:\NCI\Sem3\Thesis\implementation\data\ieee-cis\train_identity.csv")

# Merge transaction and identity data
df_ieee = pd.merge(train_transaction, train_identity, 
                   on='TransactionID', how='left')

print(f"Transaction data shape: {train_transaction.shape}")
print(f"Identity data shape:    {train_identity.shape}")
print(f"Merged shape:           {df_ieee.shape}")
print(f"Fraud rate:             {df_ieee['isFraud'].mean()*100:.2f}%")
print(f"Fraud cases:            {df_ieee['isFraud'].sum():,}")
print(f"Total transactions:     {len(df_ieee):,}")

Loading IEEE-CIS dataset...
Transaction data shape: (590540, 394)
Identity data shape:    (144233, 41)
Merged shape:           (590540, 434)
Fraud rate:             3.50%
Fraud cases:            20,663
Total transactions:     590,540


In [2]:
# ── BLOCK 2: Clean & Prepare IEEE-CIS ──

# The key column for our risk matrix
print("Transaction Amount Statistics:")
print(df_ieee['TransactionAmt'].describe())
print()

# Drop columns with too many missing values (>50% missing)
missing_pct = df_ieee.isnull().mean()
cols_to_drop = missing_pct[missing_pct > 0.5].index.tolist()
print(f"Dropping {len(cols_to_drop)} columns with >50% missing values")
df_ieee = df_ieee.drop(columns=cols_to_drop)
print(f"Shape after dropping: {df_ieee.shape}")

# Keep only numerical columns for modelling
# (removing IDs and categorical columns for simplicity)
df_ieee_num = df_ieee.select_dtypes(include=[np.number])
print(f"Numerical columns kept: {df_ieee_num.shape[1]}")

# Fill remaining missing values with median
df_ieee_num = df_ieee_num.fillna(df_ieee_num.median())

# Separate target
y_ieee = df_ieee_num['isFraud']
X_ieee = df_ieee_num.drop(columns=['isFraud', 'TransactionID'])

print(f"\nFinal feature set: {X_ieee.shape}")
print(f"Target distribution:")
print(y_ieee.value_counts())

Transaction Amount Statistics:
count    590540.000000
mean        135.027176
std         239.162522
min           0.251000
25%          43.321000
50%          68.769000
75%         125.000000
max       31937.391000
Name: TransactionAmt, dtype: float64

Dropping 214 columns with >50% missing values
Shape after dropping: (590540, 220)
Numerical columns kept: 211

Final feature set: (590540, 209)
Target distribution:
isFraud
0    569877
1     20663
Name: count, dtype: int64


In [3]:
# ── BLOCK 3: Build IEEE-CIS Risk Matrix ──

# Use percentile-based thresholds — same LOGIC as ULB
# but adapted to this dataset's distribution
p25 = df_ieee['TransactionAmt'].quantile(0.25)
p50 = df_ieee['TransactionAmt'].quantile(0.50)
p75 = df_ieee['TransactionAmt'].quantile(0.75)
p95 = df_ieee['TransactionAmt'].quantile(0.95)

print("=== IEEE-CIS Risk Matrix Thresholds ===")
print(f"Low:      $0 – ${p25:.2f}      (0-25th percentile)")
print(f"Medium:   ${p25:.2f} – ${p50:.2f}  (25-50th percentile)")
print(f"High:     ${p50:.2f} – ${p75:.2f}   (50-75th percentile)")
print(f"Critical: above ${p75:.2f}          (75th+ percentile)")
print()

def assign_risk_tier_ieee(amount):
    if amount <= p25:
        return 'Low'
    elif amount <= p50:
        return 'Medium'
    elif amount <= p75:
        return 'High'
    else:
        return 'Critical'

df_ieee['Risk_Tier'] = df_ieee['TransactionAmt'].apply(assign_risk_tier_ieee)

print("=== Risk Tier Distribution ===")
print(df_ieee['Risk_Tier'].value_counts())
print()

print("=== Fraud Rate by Risk Tier ===")
for tier in ['Low', 'Medium', 'High', 'Critical']:
    tier_data  = df_ieee[df_ieee['Risk_Tier'] == tier]
    total      = len(tier_data)
    frauds     = tier_data['isFraud'].sum()
    fraud_rate = (frauds / total) * 100
    avg_amt    = tier_data['TransactionAmt'].mean()
    print(f"{tier:10s}: {frauds:5d} frauds / {total:6d} total "
          f"({fraud_rate:.3f}%) | Avg: ${avg_amt:.2f}")

=== IEEE-CIS Risk Matrix Thresholds ===
Low:      $0 – $43.32      (0-25th percentile)
Medium:   $43.32 – $68.77  (25-50th percentile)
High:     $68.77 – $125.00   (50-75th percentile)
Critical: above $125.00          (75th+ percentile)

=== Risk Tier Distribution ===
Risk_Tier
High        149442
Low         147639
Medium      147631
Critical    145828
Name: count, dtype: int64

=== Fraud Rate by Risk Tier ===
Low       :  6264 frauds / 147639 total (4.243%) | Avg: $27.86
Medium    :  3704 frauds / 147631 total (2.509%) | Avg: $54.76
High      :  4222 frauds / 149442 total (2.825%) | Avg: $100.40
Critical  :  6473 frauds / 145828 total (4.439%) | Avg: $360.26


In [4]:
# ── BLOCK 4: Prepare IEEE-CIS for Modelling ──

# Add risk tiers to numerical dataframe
X_ieee['Risk_Tier'] = df_ieee['Risk_Tier'].values
risk_tiers_ieee = X_ieee['Risk_Tier']
X_ieee = X_ieee.drop(columns=['Risk_Tier'])

# Scale features
scaler_ieee = StandardScaler()
X_ieee_scaled = pd.DataFrame(
    scaler_ieee.fit_transform(X_ieee),
    columns=X_ieee.columns)

# Train/test split
X_tr, X_te, y_tr, y_te, rt_tr, rt_te = train_test_split(
    X_ieee_scaled, y_ieee, risk_tiers_ieee,
    test_size=0.2, random_state=42, stratify=y_ieee)

print(f"Training set: {X_tr.shape}")
print(f"Test set:     {X_te.shape}")
print(f"Fraud in train: {y_tr.sum():,}")
print(f"Fraud in test:  {y_te.sum():,}")
print()

# SMOTE
print("Applying SMOTE...")
smote = SMOTE(random_state=42)
X_tr_bal, y_tr_bal = smote.fit_resample(X_tr, y_tr)

print(f"After SMOTE - Training: {X_tr_bal.shape}")
print(f"Fraud cases: {y_tr_bal.sum():,}")
print()

# Convert to tensors
X_tr_tensor  = torch.FloatTensor(X_tr_bal.values)
y_tr_tensor  = torch.LongTensor(y_tr_bal.values)
X_te_tensor  = torch.FloatTensor(X_te.values)
y_te_tensor  = torch.LongTensor(y_te.values)

# DataLoader
tr_dataset = TensorDataset(X_tr_tensor, y_tr_tensor)
tr_loader  = DataLoader(tr_dataset, batch_size=256, shuffle=True)

print("✅ Data ready for modelling!")

Training set: (472432, 209)
Test set:     (118108, 209)
Fraud in train: 16,530
Fraud in test:  4,133

Applying SMOTE...
After SMOTE - Training: (911804, 209)
Fraud cases: 455,902

✅ Data ready for modelling!


In [5]:
# ── BLOCK 5: Define Network & Loss Functions ──

class L2DNetwork(nn.Module):
    def __init__(self, input_dim, num_classes=2):
        super(L2DNetwork, self).__init__()
        self.shared = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(64, 32),
            nn.ReLU()
        )
        self.classifier_head = nn.Linear(32, num_classes)
        self.deferral_head   = nn.Linear(32, 1)
    
    def forward(self, x):
        shared_out   = self.shared(x)
        class_logits = self.classifier_head(shared_out)
        defer_prob   = torch.sigmoid(self.deferral_head(shared_out))
        return class_logits, defer_prob

# Standard L2D loss
def l2d_loss(class_logits, defer_prob, targets, cost=0.13):
    ce_loss    = nn.CrossEntropyLoss(reduction='none')
    class_loss = ce_loss(class_logits, targets)
    defer_p    = defer_prob.squeeze()
    total_loss = ((1 - defer_p) * class_loss + defer_p * cost).mean()
    return total_loss

# Risk-Sensitive L2D loss
RISK_WEIGHTS = {
    'Low': 1.0, 'Medium': 0.8, 
    'High': 0.5, 'Critical': 0.2}
BASE_COST = 0.13

def risk_sensitive_l2d_loss(class_logits, defer_prob, 
                             targets, risk_tier_batch):
    ce_loss    = nn.CrossEntropyLoss(reduction='none')
    class_loss = ce_loss(class_logits, targets)
    defer_p    = defer_prob.squeeze()
    weights    = torch.tensor(
        [RISK_WEIGHTS[t] for t in risk_tier_batch],
        dtype=torch.float32)
    instance_cost = weights * BASE_COST
    total_loss = ((1 - defer_p) * class_loss + 
                   defer_p * instance_cost).mean()
    return total_loss

# Custom dataset with tiers
class FraudDatasetWithTiers(torch.utils.data.Dataset):
    def __init__(self, X, y, tiers):
        self.X     = X
        self.y     = y
        self.tiers = tiers
    def __len__(self):
        return len(self.y)
    def __getitem__(self, idx):
        return self.X[idx], self.y[idx], self.tiers[idx]

input_dim = X_tr_tensor.shape[1]
print(f"✅ All functions defined!")
print(f"Input dimensions: {input_dim}")

✅ All functions defined!
Input dimensions: 209


In [6]:
# ── BLOCK 6: Train All 3 Models on IEEE-CIS ──

def train_model(model, optimizer, loader, 
                num_epochs, loss_fn, loss_kwargs={}):
    losses = []
    for epoch in range(num_epochs):
        model.train()
        epoch_loss  = 0
        num_batches = 0
        for batch in loader:
            optimizer.zero_grad()
            if len(batch) == 3:
                X_b, y_b, t_b = batch
                cl, dp = model(X_b)
                loss = loss_fn(cl, dp, y_b, list(t_b))
            else:
                X_b, y_b = batch
                cl, dp = model(X_b)
                loss = loss_fn(cl, dp, y_b, **loss_kwargs)
            loss.backward()
            optimizer.step()
            epoch_loss  += loss.item()
            num_batches += 1
        avg = epoch_loss / num_batches
        losses.append(avg)
        print(f"  Epoch {epoch+1:2d}/{num_epochs} | Loss: {avg:.4f}")
    return losses

# ── MODEL 1: Baseline ──
print("=" * 50)
print("Training Model 1 — Baseline...")
print("=" * 50)
baseline_ieee = L2DNetwork(input_dim=input_dim)
opt1 = optim.Adam(baseline_ieee.parameters(), lr=0.001)

# For baseline we use standard loader and force no deferral
# by using very high cost
for epoch in range(10):
    baseline_ieee.train()
    epoch_loss = 0
    num_batches = 0
    for X_b, y_b in tr_loader:
        opt1.zero_grad()
        cl, dp = baseline_ieee(X_b)
        loss = l2d_loss(cl, dp, y_b, cost=0.99)
        loss.backward()
        opt1.step()
        epoch_loss += loss.item()
        num_batches += 1
    avg = epoch_loss / num_batches
    print(f"  Epoch {epoch+1:2d}/10 | Loss: {avg:.4f}")

print("✅ Model 1 done!")
print()

# ── MODEL 2: Standard L2D ──
print("=" * 50)
print("Training Model 2 — Standard L2D...")
print("=" * 50)
std_l2d_ieee = L2DNetwork(input_dim=input_dim)
opt2 = optim.Adam(std_l2d_ieee.parameters(), lr=0.001)

for epoch in range(10):
    std_l2d_ieee.train()
    epoch_loss = 0
    num_batches = 0
    for X_b, y_b in tr_loader:
        opt2.zero_grad()
        cl, dp = std_l2d_ieee(X_b)
        loss = l2d_loss(cl, dp, y_b, cost=0.13)
        loss.backward()
        opt2.step()
        epoch_loss += loss.item()
        num_batches += 1
    avg = epoch_loss / num_batches
    print(f"  Epoch {epoch+1:2d}/10 | Loss: {avg:.4f}")

print("✅ Model 2 done!")
print()

# ── MODEL 3: Risk-Sensitive L2D ──
print("=" * 50)
print("Training Model 3 — Risk-Sensitive L2D...")
print("=" * 50)

# Build tier array for balanced training set
rt_tr_values  = rt_tr.values
original_size = len(rt_tr_values)
smote_size    = len(X_tr_bal) - original_size
rt_tr_bal     = list(rt_tr_values) + ['Low'] * smote_size

rs_dataset_ieee = FraudDatasetWithTiers(
    X_tr_tensor, y_tr_tensor, rt_tr_bal)
rs_loader_ieee  = DataLoader(
    rs_dataset_ieee, batch_size=256, shuffle=True)

rs_l2d_ieee = L2DNetwork(input_dim=input_dim)
opt3 = optim.Adam(rs_l2d_ieee.parameters(), lr=0.001)

for epoch in range(10):
    rs_l2d_ieee.train()
    epoch_loss = 0
    num_batches = 0
    for X_b, y_b, t_b in rs_loader_ieee:
        opt3.zero_grad()
        cl, dp = rs_l2d_ieee(X_b)
        loss = risk_sensitive_l2d_loss(cl, dp, y_b, list(t_b))
        loss.backward()
        opt3.step()
        epoch_loss += loss.item()
        num_batches += 1
    avg = epoch_loss / num_batches
    print(f"  Epoch {epoch+1:2d}/10 | Loss: {avg:.4f}")

print("✅ Model 3 done!")
print()
print("🎯 All 3 models trained on IEEE-CIS!")

Training Model 1 — Baseline...
  Epoch  1/10 | Loss: 0.4463
  Epoch  2/10 | Loss: 0.3834
  Epoch  3/10 | Loss: 0.3605
  Epoch  4/10 | Loss: 0.3466
  Epoch  5/10 | Loss: 0.3369
  Epoch  6/10 | Loss: 0.3289
  Epoch  7/10 | Loss: 0.3249
  Epoch  8/10 | Loss: 0.3196
  Epoch  9/10 | Loss: 0.3153
  Epoch 10/10 | Loss: 0.3128
✅ Model 1 done!

Training Model 2 — Standard L2D...
  Epoch  1/10 | Loss: 0.1314
  Epoch  2/10 | Loss: 0.1300
  Epoch  3/10 | Loss: 0.1300
  Epoch  4/10 | Loss: 0.1300
  Epoch  5/10 | Loss: 0.1300
  Epoch  6/10 | Loss: 0.1300
  Epoch  7/10 | Loss: 0.1300
  Epoch  8/10 | Loss: 0.1300
  Epoch  9/10 | Loss: 0.1300
  Epoch 10/10 | Loss: 0.1300
✅ Model 2 done!

Training Model 3 — Risk-Sensitive L2D...
  Epoch  1/10 | Loss: 0.1072
  Epoch  2/10 | Loss: 0.1048
  Epoch  3/10 | Loss: 0.1048
  Epoch  4/10 | Loss: 0.1048
  Epoch  5/10 | Loss: 0.1048
  Epoch  6/10 | Loss: 0.1048
  Epoch  7/10 | Loss: 0.1048
  Epoch  8/10 | Loss: 0.1048
  Epoch  9/10 | Loss: 0.1048
  Epoch 10/10 | Lo

In [7]:
# ── BLOCK 7: Evaluate All 3 Models on IEEE-CIS ──

def evaluate_model(model, X_te, y_te, rt_te, model_name):
    model.eval()
    with torch.no_grad():
        cl, dp = model(X_te)
        cp     = torch.argmax(cl, dim=1).numpy()
        dp     = dp.squeeze().numpy()
        deferred   = dp > 0.5
        autonomous = ~deferred
        y_np   = y_te.values
        rt_np  = rt_te.values

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"Autonomously decided: {autonomous.sum():,} "
          f"({autonomous.mean()*100:.2f}%)")
    print(f"Deferred to human:    {deferred.sum():,} "
          f"({deferred.mean()*100:.2f}%)")
    
    if autonomous.sum() > 0:
        auto_preds  = cp[autonomous]
        auto_labels = y_np[autonomous]
        f1  = f1_score(auto_labels, auto_preds, zero_division=0)
        acc = accuracy_score(auto_labels, auto_preds)
        try:
            auc = roc_auc_score(
                auto_labels, cl[autonomous].numpy()[:,1])
        except:
            auc = 0.0
        
        print(f"F1 Score:             {f1:.4f}")
        print(f"Accuracy:             {acc:.4f}")
        print(f"ROC-AUC:              {auc:.4f}")
        
        fraud_deferred = deferred[y_np==1].sum()
        fraud_missed   = ((cp[autonomous]==0) & 
                         (y_np[autonomous]==1)).sum()
        fraud_caught   = ((cp[autonomous]==1) & 
                         (y_np[autonomous]==1)).sum()
        print(f"Frauds deferred:      {fraud_deferred}")
        print(f"Frauds caught:        {fraud_caught}")
        print(f"Frauds missed:        {fraud_missed}")
    
    print(f"\nDeferral by Risk Tier:")
    tier_rates = {}
    for tier in ['Low','Medium','High','Critical']:
        mask  = rt_np == tier
        rate  = deferred[mask].mean() * 100
        tier_rates[tier] = rate
        print(f"  {tier:10s}: {rate:.2f}%")
    
    return tier_rates

# Run evaluation
X_te_tensor_ieee = torch.FloatTensor(X_te.values)

print("EVALUATING ALL 3 MODELS ON IEEE-CIS...")
r1 = evaluate_model(baseline_ieee,  X_te_tensor_ieee, 
                    y_te, rt_te, "MODEL 1 — BASELINE")
r2 = evaluate_model(std_l2d_ieee,   X_te_tensor_ieee, 
                    y_te, rt_te, "MODEL 2 — STANDARD L2D")
r3 = evaluate_model(rs_l2d_ieee,    X_te_tensor_ieee, 
                    y_te, rt_te, "MODEL 3 — RISK-SENSITIVE L2D")

print(f"\n{'='*55}")
print("KEY COMPARISON — DEFERRAL BY RISK TIER")
print(f"{'='*55}")
print(f"{'Tier':<12} {'Model 2':>10} {'Model 3':>10} {'Change':>10}")
print("-" * 45)
for tier in ['Low','Medium','High','Critical']:
    change = r3[tier] - r2[tier]
    direction = "⬆" if change > 0 else "⬇"
    print(f"  {tier:<10} {r2[tier]:>9.2f}% {r3[tier]:>9.2f}%"
          f" {direction}{abs(change):>6.2f}%")

EVALUATING ALL 3 MODELS ON IEEE-CIS...

  MODEL 1 — BASELINE
Autonomously decided: 118,108 (100.00%)
Deferred to human:    0 (0.00%)
F1 Score:             0.3276
Accuracy:             0.8962
ROC-AUC:              0.8829
Frauds deferred:      0
Frauds caught:        2988
Frauds missed:        1145

Deferral by Risk Tier:
  Low       : 0.00%
  Medium    : 0.00%
  High      : 0.00%
  Critical  : 0.00%

  MODEL 2 — STANDARD L2D
Autonomously decided: 0 (0.00%)
Deferred to human:    118,108 (100.00%)

Deferral by Risk Tier:
  Low       : 100.00%
  Medium    : 100.00%
  High      : 100.00%
  Critical  : 100.00%

  MODEL 3 — RISK-SENSITIVE L2D
Autonomously decided: 0 (0.00%)
Deferred to human:    118,108 (100.00%)

Deferral by Risk Tier:
  Low       : 100.00%
  Medium    : 100.00%
  High      : 100.00%
  Critical  : 100.00%

KEY COMPARISON — DEFERRAL BY RISK TIER
Tier            Model 2    Model 3     Change
---------------------------------------------
  Low           100.00%    100.00% ⬇  0.

In [8]:
# ── BLOCK 8: Find Right Deferral Cost for IEEE-CIS ──

print("Finding optimal deferral cost for IEEE-CIS...")
print("=" * 50)

costs_to_try = [0.3, 0.5, 0.7]

for cost_val in costs_to_try:
    m = L2DNetwork(input_dim=input_dim)
    opt = optim.Adam(m.parameters(), lr=0.001)
    
    for epoch in range(10):
        m.train()
        for X_b, y_b in tr_loader:
            opt.zero_grad()
            cl, dp = m(X_b)
            loss = l2d_loss(cl, dp, y_b, cost=cost_val)
            loss.backward()
            opt.step()
    
    # Quick evaluation
    m.eval()
    with torch.no_grad():
        cl, dp = m(X_te_tensor_ieee)
        dp     = dp.squeeze().numpy()
        deferred = dp > 0.5
        y_np   = y_te.values
        rt_np  = rt_te.values
    
    deferral_rate = deferred.mean() * 100
    
    # Tier rates
    tier_rates = {}
    for tier in ['Low','Medium','High','Critical']:
        mask = rt_np == tier
        tier_rates[tier] = deferred[mask].mean() * 100
    
    print(f"Cost={cost_val} | Overall: {deferral_rate:.1f}% | "
          f"Low: {tier_rates['Low']:.1f}% | "
          f"Critical: {tier_rates['Critical']:.1f}%")

print()
print("Looking for cost where overall deferral is between 1-10%")

Finding optimal deferral cost for IEEE-CIS...
Cost=0.3 | Overall: 98.6% | Low: 97.1% | Critical: 99.2%
Cost=0.5 | Overall: 44.0% | Low: 37.7% | Critical: 59.1%
Cost=0.7 | Overall: 0.0% | Low: 0.0% | Critical: 0.0%

Looking for cost where overall deferral is between 1-10%


In [9]:
# ── BLOCK 9: Fine-tune Cost for IEEE-CIS ──

costs_to_try = [0.55, 0.60, 0.63, 0.65]

for cost_val in costs_to_try:
    m = L2DNetwork(input_dim=input_dim)
    opt = optim.Adam(m.parameters(), lr=0.001)
    
    for epoch in range(10):
        m.train()
        for X_b, y_b in tr_loader:
            opt.zero_grad()
            cl, dp = m(X_b)
            loss = l2d_loss(cl, dp, y_b, cost=cost_val)
            loss.backward()
            opt.step()
    
    m.eval()
    with torch.no_grad():
        cl, dp = m(X_te_tensor_ieee)
        cp     = torch.argmax(cl, dim=1).numpy()
        dp     = dp.squeeze().numpy()
        deferred   = dp > 0.5
        autonomous = ~deferred
        y_np   = y_te.values
        rt_np  = rt_te.values
    
    deferral_rate = deferred.mean() * 100
    
    if autonomous.sum() > 0:
        f1 = f1_score(y_np[autonomous], 
                      cp[autonomous], zero_division=0)
    else:
        f1 = 0
    
    tier_rates = {}
    for tier in ['Low','Medium','High','Critical']:
        mask = rt_np == tier
        tier_rates[tier] = deferred[mask].mean() * 100
    
    print(f"Cost={cost_val} | Deferral: {deferral_rate:.1f}% | "
          f"F1: {f1:.4f} | "
          f"Low: {tier_rates['Low']:.1f}% | "
          f"Critical: {tier_rates['Critical']:.1f}%")

Cost=0.55 | Deferral: 31.0% | F1: 0.5615 | Low: 27.2% | Critical: 40.7%
Cost=0.6 | Deferral: 21.2% | F1: 0.4979 | Low: 20.4% | Critical: 27.8%
Cost=0.63 | Deferral: 6.7% | F1: 0.3906 | Low: 13.4% | Critical: 4.2%
Cost=0.65 | Deferral: 0.0% | F1: 0.3034 | Low: 0.0% | Critical: 0.0%


In [10]:
# ── BLOCK 10: Train Final Models for IEEE-CIS ──
# NOTE: corrected during verification pass (22-23 Jul 2026). Previously Model 2
# used base cost 0.63 while Model 3 used 0.67 -- two variables changed at once,
# confounding the comparison. Both now share IEEE_BASE_COST = 0.63, and model
# training is seeded (torch.manual_seed) for reproducibility. See Master
# Reference Document Section 11 for the full explanation.

IEEE_BASE_COST = 0.63

# Risk weights (dataset-specific: gentler than ULB because IEEE-CIS already
# has a much higher base fraud rate)
IEEE_RISK_WEIGHTS = {
    'Low': 1.0, 'Medium': 0.9,
    'High': 0.7, 'Critical': 0.5}

print("Training Final Models for IEEE-CIS...")
print()

# ── MODEL 1: Baseline (high cost = never defers) ──
print("=" * 50)
print("Model 1 -- Baseline...")
torch.manual_seed(42)
baseline_ieee_final = L2DNetwork(input_dim=input_dim)
opt1 = optim.Adam(baseline_ieee_final.parameters(), lr=0.001)
loader1 = DataLoader(TensorDataset(X_tr_tensor, y_tr_tensor), batch_size=256,
                      shuffle=True, generator=torch.Generator().manual_seed(42))

for epoch in range(15):
    baseline_ieee_final.train()
    epoch_loss = 0
    num_batches = 0
    for X_b, y_b in loader1:
        opt1.zero_grad()
        cl, dp = baseline_ieee_final(X_b)
        loss = l2d_loss(cl, dp, y_b, cost=0.99)
        loss.backward()
        opt1.step()
        epoch_loss += loss.item()
        num_batches += 1
    avg = epoch_loss / num_batches
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/15 | Loss: {avg:.4f}")
print("✅ Baseline done!")
print()

# ── MODEL 2: Standard L2D (uniform cost = IEEE_BASE_COST) ──
print("=" * 50)
print("Model 2 -- Standard L2D...")
torch.manual_seed(42)
std_ieee_final = L2DNetwork(input_dim=input_dim)
opt2 = optim.Adam(std_ieee_final.parameters(), lr=0.001)
loader2 = DataLoader(TensorDataset(X_tr_tensor, y_tr_tensor), batch_size=256,
                      shuffle=True, generator=torch.Generator().manual_seed(42))

for epoch in range(15):
    std_ieee_final.train()
    epoch_loss = 0
    num_batches = 0
    for X_b, y_b in loader2:
        opt2.zero_grad()
        cl, dp = std_ieee_final(X_b)
        loss = l2d_loss(cl, dp, y_b, cost=IEEE_BASE_COST)
        loss.backward()
        opt2.step()
        epoch_loss += loss.item()
        num_batches += 1
    avg = epoch_loss / num_batches
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/15 | Loss: {avg:.4f}")
print("✅ Standard L2D done!")
print()

# ── MODEL 3: Risk-Sensitive L2D (same base cost as Model 2) ──
print("=" * 50)
print("Model 3 -- Risk-Sensitive L2D...")

def rs_loss_ieee(class_logits, defer_prob, targets, risk_tier_batch):
    ce_loss    = nn.CrossEntropyLoss(reduction='none')
    class_loss = ce_loss(class_logits, targets)
    defer_p    = defer_prob.squeeze()
    weights    = torch.tensor(
        [IEEE_RISK_WEIGHTS[t] for t in risk_tier_batch],
        dtype=torch.float32)
    instance_cost = weights * IEEE_BASE_COST
    return ((1-defer_p)*class_loss + defer_p*instance_cost).mean()

torch.manual_seed(42)
rs_ieee_final = L2DNetwork(input_dim=input_dim)
opt3 = optim.Adam(rs_ieee_final.parameters(), lr=0.001)
rs_loader_ieee = DataLoader(rs_dataset_ieee, batch_size=256, shuffle=True,
                             generator=torch.Generator().manual_seed(42))

for epoch in range(15):
    rs_ieee_final.train()
    epoch_loss = 0
    num_batches = 0
    for X_b, y_b, t_b in rs_loader_ieee:
        opt3.zero_grad()
        cl, dp = rs_ieee_final(X_b)
        loss = rs_loss_ieee(cl, dp, y_b, list(t_b))
        loss.backward()
        opt3.step()
        epoch_loss += loss.item()
        num_batches += 1
    avg = epoch_loss / num_batches
    if (epoch+1) % 5 == 0:
        print(f"  Epoch {epoch+1:2d}/15 | Loss: {avg:.4f}")
print("✅ Risk-Sensitive L2D done!")
print()
print("All 3 final models trained on IEEE-CIS (base cost harmonised at 0.63)!")

Training Final Models for IEEE-CIS...

Model 1 -- Baseline...
  Epoch  5/15 | Loss: 0.3407
  Epoch 10/15 | Loss: 0.3158
  Epoch 15/15 | Loss: 0.3045
✅ Baseline done!

Model 2 -- Standard L2D...
  Epoch  5/15 | Loss: 0.3364
  Epoch 10/15 | Loss: 0.3143
  Epoch 15/15 | Loss: 0.3030
✅ Standard L2D done!

Model 3 -- Risk-Sensitive L2D...
  Epoch  5/15 | Loss: 0.3401
  Epoch 10/15 | Loss: 0.3209
  Epoch 15/15 | Loss: 0.3102
✅ Risk-Sensitive L2D done!

All 3 final models trained on IEEE-CIS (base cost harmonised at 0.63)!

In [11]:
# ── BLOCK 11: Final Evaluation on IEEE-CIS ──

def full_evaluate(model, X_te, y_te, rt_te, model_name):
    model.eval()
    with torch.no_grad():
        cl, dp = model(X_te)
        cp     = torch.argmax(cl, dim=1).numpy()
        dp     = dp.squeeze().numpy()
        deferred   = dp > 0.5
        autonomous = ~deferred
        y_np   = y_te.values
        rt_np  = rt_te.values

    print(f"\n{'='*55}")
    print(f"  {model_name}")
    print(f"{'='*55}")
    print(f"Autonomously decided: {autonomous.sum():,} "
          f"({autonomous.mean()*100:.2f}%)")
    print(f"Deferred to human:    {deferred.sum():,} "
          f"({deferred.mean()*100:.2f}%)")

    if autonomous.sum() > 0:
        auto_preds  = cp[autonomous]
        auto_labels = y_np[autonomous]
        f1  = f1_score(auto_labels, auto_preds, zero_division=0)
        acc = accuracy_score(auto_labels, auto_preds)
        try:
            auc = roc_auc_score(
                auto_labels, cl[autonomous].numpy()[:,1])
        except:
            auc = 0.0
        print(f"F1 Score:             {f1:.4f}")
        print(f"Accuracy:             {acc:.4f}")
        print(f"ROC-AUC:              {auc:.4f}")
        fraud_deferred = deferred[y_np==1].sum()
        fraud_caught   = ((cp[autonomous]==1) &
                         (y_np[autonomous]==1)).sum()
        fraud_missed   = ((cp[autonomous]==0) &
                         (y_np[autonomous]==1)).sum()
        print(f"Frauds deferred:      {fraud_deferred:,}")
        print(f"Frauds caught:        {fraud_caught:,}")
        print(f"Frauds missed:        {fraud_missed:,}")

    print(f"\nDeferral by Risk Tier:")
    tier_rates = {}
    for tier in ['Low','Medium','High','Critical']:
        mask  = rt_np == tier
        rate  = deferred[mask].mean() * 100
        tier_rates[tier] = rate
        print(f"  {tier:10s}: {rate:.2f}%")

    return tier_rates

r1 = full_evaluate(baseline_ieee_final, X_te_tensor_ieee,
                   y_te, rt_te, "MODEL 1 — BASELINE")
r2 = full_evaluate(std_ieee_final,      X_te_tensor_ieee,
                   y_te, rt_te, "MODEL 2 — STANDARD L2D")
r3 = full_evaluate(rs_ieee_final,       X_te_tensor_ieee,
                   y_te, rt_te, "MODEL 3 — RISK-SENSITIVE L2D")

print(f"\n{'='*55}")
print("KEY COMPARISON — DEFERRAL BY RISK TIER")
print(f"{'='*55}")
print(f"{'Tier':<12} {'Model 2':>10} {'Model 3':>10} {'Change':>10}")
print("-" * 45)
for tier in ['Low','Medium','High','Critical']:
    change    = r3[tier] - r2[tier]
    direction = "⬆" if change > 0 else "⬇"
    print(f"  {tier:<10} {r2[tier]:>9.2f}% "
          f"{r3[tier]:>9.2f}% "
          f"{direction}{abs(change):>6.2f}%")

print()
print("ULB REFERENCE:")
print("  Critical tier: 0.37% → 5.45% (14.7x increase)")
print("  Low tier:      0.46% → 0.52% (barely changed)")


  MODEL 1 -- BASELINE
Autonomously decided: 118,108 (100.00%)
Deferred to human:    0 (0.00%)
F1 Score:             0.3426
Accuracy:             0.8992
ROC-AUC:              0.9046
Frauds deferred:      0
Frauds caught:        3,103
Frauds missed:        1,030

Deferral by Risk Tier:
  Low       : 0.00%
  Medium    : 0.00%
  High      : 0.00%
  Critical  : 0.00%

  MODEL 2 -- STANDARD L2D
Autonomously decided: 107,582 (91.09%)
Deferred to human:    10,526 (8.91%)
F1 Score:             0.4001
Accuracy:             0.9185
ROC-AUC:              0.9128
Frauds deferred:      307
Frauds caught:        2,923
Frauds missed:        903

Deferral by Risk Tier:
  Low       : 12.74%
  Medium    : 3.60%
  High      : 8.68%
  Critical  : 10.67%

  MODEL 3 -- RISK-SENSITIVE L2D
Autonomously decided: 75,475 (63.90%)
Deferred to human:    42,633 (36.10%)
F1 Score:             0.5593
Accuracy:             0.9533
ROC-AUC:              0.9435
Frauds deferred:      1,437
Frauds caught:        2,236
Frauds

In [ ]:
# NOTE: superseded during verification pass (22-23 Jul 2026).
# The base-cost harmonisation fix (IEEE_BASE_COST = 0.63 for both Model 2 and
# Model 3, IEEE_RISK_WEIGHTS as defined) is now applied directly in the
# 'Train Final Models' cell above. This cell's original exploratory retrain
# (different weight/cost combinations) is kept as a comment for the record but
# no longer re-executed, since running it again would just reproduce an
# intermediate step on the way to the corrected final configuration used above.


In [ ]:
# NOTE: superseded during verification pass (22-23 Jul 2026).
# The base-cost harmonisation fix (IEEE_BASE_COST = 0.63 for both Model 2 and
# Model 3, IEEE_RISK_WEIGHTS as defined) is now applied directly in the
# 'Train Final Models' cell above. This cell's original exploratory retrain
# (different weight/cost combinations) is kept as a comment for the record but
# no longer re-executed, since running it again would just reproduce an
# intermediate step on the way to the corrected final configuration used above.


In [14]:
# ── BLOCK 14: Save IEEE-CIS Results ──
import json
import os

ieee_results = {
    'dataset': 'IEEE-CIS Fraud Detection',
    'total_test': 118108,
    'fraud_rate': 3.50,
    'base_cost': IEEE_BASE_COST,
    'risk_weights': IEEE_RISK_WEIGHTS,
    'seed': 42,
    'model1_baseline': {
        'deferral_rate': 0.0, 'f1_score': 0.3426, 'roc_auc': 0.9046,
        'frauds_missed': 1030,
        'deferral_by_tier': {'Low': 0.0, 'Medium': 0.0, 'High': 0.0, 'Critical': 0.0}
    },
    'model2_standard_l2d': {
        'deferral_rate': 8.91, 'f1_score': 0.4001, 'roc_auc': 0.9128,
        'frauds_missed': 903, 'frauds_deferred': 307,
        'deferral_by_tier': {'Low': 12.74, 'Medium': 3.60, 'High': 8.68, 'Critical': 10.67}
    },
    'model3_risk_sensitive': {
        'deferral_rate': 36.10, 'f1_score': 0.5593, 'roc_auc': 0.9435,
        'frauds_missed': 460, 'frauds_deferred': 1437,
        'deferral_by_tier': {'Low': 18.60, 'Medium': 13.62, 'High': 41.94, 'Critical': 70.56},
        'critical_low_ratio': 3.8
    },
    'key_finding': 'Critical tier defers 3.8x more than Low tier'
}

os.makedirs(
    r"E:\NCI\Sem3\Thesis\implementation\results", exist_ok=True)
with open(
    r"E:\NCI\Sem3\Thesis\implementation\results\ieee_cis_results.json",
    'w') as f:
    json.dump(ieee_results, f, indent=4)

print("✅ IEEE-CIS results saved!")
print()
print("=== CROSS-DATASET SUMMARY SO FAR ===")
print()
print(f"{'Dataset':<15} {'Critical Tier':>15} {'Low Tier':>10} {'Ratio':>8}")
print("-" * 52)
print(f"{'ULB':<15} {'0.96%':>15} {'0.42%':>10} {'4.6x':>8}")
print(f"{'IEEE-CIS':<15} {'70.56%':>15} {'18.60%':>10} {'3.8x':>8}")
print()
print("Both datasets show Critical tier deferring")
print("significantly more than Low tier -- with Model 2 (uniform cost)")
print("under-deferring Critical relative to Low on IEEE-CIS specifically.")
print()
print("Next: PaySim validation")

✅ IEEE-CIS results saved!

=== CROSS-DATASET SUMMARY SO FAR ===

Dataset           Critical Tier   Low Tier    Ratio
----------------------------------------------------
ULB                       0.96%      0.42%     4.6x
IEEE-CIS                 70.56%     18.60%     3.8x

Both datasets show Critical tier deferring
significantly more than Low tier -- with Model 2 (uniform cost)
under-deferring Critical relative to Low on IEEE-CIS specifically.

Next: PaySim validation